# 03. Preprocesamiento de Interacciones

A partir del notebook de referencia, este notebook deja listas las interacciones para modelos de recomendación y evaluación temporal.


In [1]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings('ignore')

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 5)

from src.utils.recsys_utils import (
    add_indices,
    build_interactions,
    create_index_maps,
    ensure_dir,
    filter_min_interactions,
    load_clean_data,
    popularity_ranking,
    temporal_split,
)

In [2]:
_, movielens = load_clean_data(source="movielens", use_cache=True)
processed_dir = ensure_dir(ROOT / 'data' / 'processed')
tables_dir = ensure_dir(ROOT / 'reports' / 'tables')

[20:51:07] INFO - Iniciando pipeline de carga y limpieza | source=movielens | use_cache=True | refresh_cache=False
[20:51:07] INFO - Cargando cache limpia para movielens
[20:51:07] INFO - Cache cargada: links.pkl | filas=87585 columnas=4
[20:51:07] INFO - Cache cargada: movies.pkl | filas=87585 columnas=6
[20:51:07] INFO - Cache cargada: ratings.pkl | filas=32000204 columnas=4
[20:51:07] INFO - Cache cargada: tags.pkl | filas=2000072 columnas=4
[20:51:07] INFO - Se omite carga de IMDb porque no fue solicitado
[20:51:07] INFO - Pipeline de carga y limpieza completado | source=movielens


## Construir interacciones limpias

Se parte de `ratings.csv`, se aplican filtros mínimos y se prepara un split temporal global.


In [3]:
interactions = build_interactions(movielens)
raw_summary = pd.DataFrame([
    {'metric': 'raw_rows', 'value': len(interactions)},
    {'metric': 'raw_users', 'value': interactions['userId'].nunique()},
    {'metric': 'raw_items', 'value': interactions['movieId'].nunique()},
])
display(raw_summary)

filtered = filter_min_interactions(interactions, min_user_ratings=20, min_item_ratings=20, iterations=3)
filtered_summary = pd.DataFrame([
    {'metric': 'filtered_rows', 'value': len(filtered)},
    {'metric': 'filtered_users', 'value': filtered['userId'].nunique()},
    {'metric': 'filtered_items', 'value': filtered['movieId'].nunique()},
])
display(filtered_summary)

[20:51:28] INFO - Interacciones preparadas | filas=32000204


,metric,value
0,raw_rows,32000204
1,raw_users,200948
2,raw_items,84432


[20:51:28] INFO - Filtrando interacciones | min_user_ratings=20 | min_item_ratings=20 | iterations=3
[20:51:30] INFO - Filtro iteracion 1/3 | filas=31725920 usuarios=200948 items=23350
[20:51:31] INFO - Filtro iteracion 2/3 | filas=31722400 usuarios=200763 items=23339
[20:51:32] INFO - Filtro iteracion 3/3 | filas=31722400 usuarios=200763 items=23339


,metric,value
0,filtered_rows,31722400
1,filtered_users,200763
2,filtered_items,23339


## Split temporal y mapeos de índices

El split respeta el orden temporal global para evitar leakage en la evaluación.


In [4]:
train, valid, test = temporal_split(filtered, train_ratio=0.8, valid_ratio=0.1)
user_index, item_index = create_index_maps(train)

train_idx = add_indices(train, user_index, item_index)
valid_idx = add_indices(valid[valid['userId'].isin(user_index['userId']) & valid['movieId'].isin(item_index['movieId'])], user_index, item_index)
test_idx = add_indices(test[test['userId'].isin(user_index['userId']) & test['movieId'].isin(item_index['movieId'])], user_index, item_index)

split_summary = pd.DataFrame([
    {'split': 'train', 'rows': len(train_idx), 'users': train_idx['userId'].nunique(), 'items': train_idx['movieId'].nunique()},
    {'split': 'valid', 'rows': len(valid_idx), 'users': valid_idx['userId'].nunique(), 'items': valid_idx['movieId'].nunique()},
    {'split': 'test', 'rows': len(test_idx), 'users': test_idx['userId'].nunique(), 'items': test_idx['movieId'].nunique()},
])
display(split_summary)


[20:51:38] INFO - Generando split temporal | train_ratio=0.80 | valid_ratio=0.10
[20:51:41] INFO - Split temporal listo | train=25377920 valid=3172240 test=3172240
[20:51:41] INFO - Creando mapas de indices de usuarios e items
[20:51:41] INFO - Mapas creados | users=169847 | items=20943


,split,rows,users,items
0,train,25377920,169847,20943
1,valid,457592,6637,19183
2,test,301839,4493,18407


## Baseline de popularidad y persistencia de artefactos

Además del split, se guarda un baseline simple para comparar luego contra CF e híbrido.


In [5]:
popularity = popularity_ranking(train_idx)
display(popularity.head(10))

train_idx.to_parquet(processed_dir / 'interactions_train.parquet', index=False)
valid_idx.to_parquet(processed_dir / 'interactions_valid.parquet', index=False)
test_idx.to_parquet(processed_dir / 'interactions_test.parquet', index=False)
user_index.to_parquet(processed_dir / 'user_index.parquet', index=False)
item_index.to_parquet(processed_dir / 'item_index.parquet', index=False)
popularity.to_parquet(processed_dir / 'popularity_baseline.parquet', index=False)

preprocessing_summary = pd.concat([raw_summary, filtered_summary], ignore_index=True)
preprocessing_summary.to_csv(tables_dir / 'interaction_preprocessing_summary.csv', index=False)
split_summary.to_csv(tables_dir / 'interaction_split_summary.csv', index=False)


,movieId,popularity_score,rating_count
0,356,4.043553,84724
1,318,4.417307,83701
2,296,4.183157,82757
3,593,4.151844,77290
4,2571,4.150244,73903
5,260,4.129807,71117
6,480,3.672380,68320
7,527,4.255003,62860
8,110,4.004448,62617
9,589,3.940854,60528
